# tuned - Kaggle smoke run
Prereqs: phone-verified account, Accelerator = **GPU T4 x2** (never P100), Internet **On**,
`HF_TOKEN` added under Add-ons -> Secrets. Set `MODE` below, then Run All
(SAVETEST interactively first; SMOKE via *Save & Run All* in the background).

In [ ]:
MODE = "SAVETEST"  # SAVETEST (4-step save/push gate) | SMOKE (60 steps) | RESUME

import os, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single-T4 training (spec: DDP deferred)
os.environ["HF_HOME"] = "/tmp/hf_cache"          # scratch, NOT the 20GB persisted /kaggle/working
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpus)
assert gpus.count("T4") == 2, "Expected 2x T4 - Settings -> Accelerator -> 'GPU T4 x2'"
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
%cd /tmp
!rm -rf /tmp/tuned
!git clone --depth 1 https://github.com/Anant-T/Tuned /tmp/tuned
%cd /tmp/tuned

In [ ]:
!pip install -q uv
!uv pip install --system -e ".[dev,train]"

In [ ]:
import os
from pathlib import Path

tok = None
for f in Path("/kaggle/input").glob("*/token.txt"):
    tok = f.read_text().strip()
    break
if tok is None:
    from kaggle_secrets import UserSecretsClient

    tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = tok
print("HF token loaded (not printed).")

In [ ]:
from importlib.metadata import version

for pkg in ("torch", "transformers", "trl", "unsloth", "bitsandbytes", "peft", "hf_transfer"):
    try:
        print(f"{pkg}=={version(pkg)}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

import subprocess

assert subprocess.run(["python", "-m", "pytest", "tests/", "-q"]).returncode == 0, "tests failed - fix before burning GPU quota"

In [ ]:
import subprocess

assert subprocess.run(["python", "-m", "tuned.data.smoke", "--config", "configs/law_v1.yaml"]).returncode == 0, "dataset build failed"

In [ ]:
CONFIG = "configs/law_v1.yaml"  # escape hatch: configs/law_v1_qwen.yaml (see runbook)

if MODE == "SAVETEST":
    !python -m tuned.train.sft --config {CONFIG} --mode smoke --max-steps 4 --save-steps 2
elif MODE == "SMOKE":
    !python -m tuned.train.sft --config {CONFIG} --mode smoke
elif MODE == "RESUME":
    !python -m tuned.train.sft --config {CONFIG} --mode smoke --resume
else:
    raise ValueError(f"unknown MODE {MODE!r}")

## Green means
- **SAVETEST**: no `# of LoRAs ... does not match` error (unsloth#5677); `last-checkpoint/`
  visible in the private HF checkpoint repo. If it fails twice after the regex scoping,
  switch `CONFIG` above to `configs/law_v1_qwen.yaml` (see runbook in the plan doc).
- **SMOKE**: 60 steps complete, loss trending down, **no NaN** (fp16 canary),
  `peak_vram_gb` < 14. Expected duration 4-6 h - record `approx_tokens_per_sec`
  and total session hours for the main-run plan.
- **RESUME**: run in a *fresh* session; training continues from step 25/50, not step 0.
- Note: SAVETEST touches only ~64 examples - an OOM later in the full SMOKE run is still possible; watch peak_vram_gb.